→ Technique #4: DAPT + SFT + RAG

This notebook:

Uses FAISS to retrieve university knowledge

Loads both LoRA adapters: DAPT + QLoRA

Feeds the retrieved context + question into the merged model

Returns the final answer

🧠 Purpose:
This script implements Pipeline P4: Full Stack — DAPT + SFT + RAG, using:

FAISS for context retrieval

LoRA-DAPT adapter for domain pretraining

QLoRA SFT adapter for instruction-following

LLaMA 2 base in 4-bit quantized mode

It is your final production chatbot pipeline.

Step 1: Imports & Paths
Prepares all libraries: FAISS, PEFT, transformers, sentence transformers.

Defines the file paths for:

LLaMA base

DAPT and QLoRA adapters

FAISS index and precomputed chunks

In [2]:
import torch
import faiss
import pickle
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Paths
base_model_id = "meta-llama/Llama-2-7b-hf"
dapt_adapter_path = "../checkpoints/llama2_dapt_lora/"
sft_adapter_path = "../checkpoints/qlora_finetuned_model/"
index_path = "../checkpoints/faiss_embeddings/university_index.faiss"
chunk_path = "../checkpoints/faiss_embeddings/university_chunks.pkl"


Step 2: Load Tokenizer
Loads the tokenizer and assigns pad token.

In [3]:
tokenizer = AutoTokenizer.from_pretrained(base_model_id, use_auth_token=True)
tokenizer.pad_token = tokenizer.eos_token


c:\Users\berfi\anaconda3\envs\ml_env\lib\site-packages\transformers\models\auto\tokenization_auto.py:809: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Step 3: Load Base Model + Merge Adapters
Loads base model in 4-bit

Applies DAPT LoRA adapter first

Applies SFT (QLoRA) adapter on top — creating a merged dual-adapter model

In [4]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    use_auth_token=True
)

# Apply DAPT adapter
model = PeftModel.from_pretrained(base_model, dapt_adapter_path)

# Apply QLoRA (SFT) adapter on top
model = PeftModel.from_pretrained(model, sft_adapter_path)
model.eval()


c:\Users\berfi\anaconda3\envs\ml_env\lib\site-packages\transformers\models\auto\auto_factory.py:471: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): PeftModelForCausalLM(
      (base_model): LoraModel(
        (model): LlamaForCausalLM(
          (model): LlamaModel(
            (embed_tokens): Embedding(32000, 4096)
            (layers): ModuleList(
              (0-31): 32 x LlamaDecoderLayer(
                (self_attn): LlamaSdpaAttention(
                  (q_proj): lora.Linear4bit(
                    (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                    (lora_dropout): ModuleDict(
                      (default): Dropout(p=0.05, inplace=False)
                    )
                    (lora_A): ModuleDict(
                      (default): Linear(in_features=4096, out_features=8, bias=False)
                    )
                    (lora_B): ModuleDict(
                      (default): Linear(in_features=8, out_features=4096, bias=False)
                    )
                    (lora_embedding_A): ParameterDict()
          

Step 4: Load FAISS + Text Chunks
Loads:

university_index.faiss: the semantic vector index

university_chunks.pkl: the original text passages

Embedding model (BAAI/bge-small-en-v1.5) is used to encode questions for retrieval

In [5]:
index = faiss.read_index(index_path)

with open(chunk_path, "rb") as f:
    chunks = pickle.load(f)

embedding_model = SentenceTransformer("BAAI/bge-small-en-v1.5")


Step 5: FAISS Search Helper
Retrieves the top-k most relevant chunk(s) from the FAISS index using the embedded query.



In [6]:
def search_faiss(query, top_k=1):
    embedded = embedding_model.encode([query])
    D, I = index.search(np.array(embedded), top_k)
    return [chunks[i] for i in I[0]]


Step 6: ask_full_chatbot()
Injects the FAISS-retrieved context and the question into a natural prompt

Runs model.generate() to answer



In [7]:
def ask_full_chatbot(question):
    context = search_faiss(question, top_k=1)[0]

    prompt = f"""You are a helpful assistant at a university.

Use the following university information to answer the question clearly.

Context: {context}

Question: {question}

Answer:"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=True,
            temperature=0.7,
            top_k=50,
            top_p=0.95
        )

    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer.split("Answer:")[-1].strip()


Step 7: Try It
Sends 4 standard questions

Gets domain-specific, RAG-informed, fluently structured responses

In [8]:
questions = [
    "Where can I book a study room?",
    "What is the PGR Lounge?",
    "How do I contact IT services?",
    "Can I get help with academic writing?",
]

for q in questions:
    print(f"🧑‍🎓 Question: {q}")
    print("🤖 Answer:", ask_full_chatbot(q))
    print("-" * 80)


🧑‍🎓 Question: Where can I book a study room?


c:\Users\berfi\anaconda3\envs\ml_env\lib\site-packages\transformers\models\bert\modeling_bert.py:440: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


🤖 Answer: Study Rooms are available to book via Resource Booker. They are available on a first-come-first-served basis and must be booked at least 30 minutes in advance.

You can book a room for a maximum of 3 hours.

You can book a maximum of 3 rooms at the same time.

Please note that rooms are designed for a maximum of 6 people.

If you are booking more than one room you must leave at least 30 minutes between bookings to allow for room set up and clearing.

If you arrive early, you may use the room for an hour or two before your booking starts.
--------------------------------------------------------------------------------
🧑‍🎓 Question: What is the PGR Lounge?
🤖 Answer: The PGR Lounge is an informal peer support and study space for PGRs at the University of York. The PGR Lounge is available for PGRs to use during office hours. The lounge is equipped with computers, printers, scanners, and a tea/coffee station. PGRs can access the lounge by signing in at the reception desk or by sca

✅ Suggestions

Area	Suggestion
File name	✅ Keep as rag_dapt_sft_chatbot.ipynb or rename to university_chatbot_full.ipynb
Prompt	Consider putting prompt template in a .txt file or variable for reuse
Logging	Save responses to file or display side-by-side with other pipelines
Inference API	Eventually wrap in a Flask or Gradio app if deployment is planned
✅ Pipeline Reference

Pipeline	Description
P4	✅ DAPT + SFT + RAG (final system)